In [14]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import lightgbm as lgb

pd.set_option("display.max_columns", None)

In [15]:
!ls ../artifacts/datasets/

dataset_level_01_daily_total.parquet
dataset_level_02_daily_state.parquet
dataset_level_03_daily_cat.parquet
dataset_level_04_daily_dept.parquet
dataset_level_05_daily_state_cat.parquet
dataset_level_06_daily_store.parquet
dataset_level_07_daily_state_dept.parquet
dataset_level_08_daily_store_cat.parquet
dataset_level_09_daily_store_dept.parquet
dataset_level_10_weekly_item.parquet
dataset_level_11_weekly_item_state.parquet
dataset_level_12_weekly_item_store.parquet


In [16]:
results_df = pd.read_parquet('../output/experiment_results.parquet')
results_df.sort_values('wape').head(10)

,level_id,level_name,grain,target,target_type,split,horizon,n_series,n_rows,fit_time_s,wape,bias,wrmsse,model_path
26,1,total,daily,cum365,cumulative,valid,365,1,1576,0.03,0.024306,-0.024306,NaN,/home/njimenez/Workspace/magister/forecast-tfm...
24,1,total,daily,cum42,cumulative,valid,365,1,1899,0.12,0.026475,0.021023,NaN,/home/njimenez/Workspace/magister/forecast-tfm...
22,1,total,daily,cum35,cumulative,valid,365,1,1906,0.09,0.028751,0.024571,NaN,/home/njimenez/Workspace/magister/forecast-tfm...
20,1,total,daily,cum28,cumulative,valid,365,1,1913,0.06,0.030585,0.023946,NaN,/home/njimenez/Workspace/magister/forecast-tfm...
2,1,total,daily,sales,sales,valid,21,1,1941,0.12,0.033788,0.007337,0.303125,/home/njimenez/Workspace/magister/forecast-tfm...
18,1,total,daily,cum21,cumulative,valid,365,1,1920,0.11,0.033918,0.024390,NaN,/home/njimenez/Workspace/magister/forecast-tfm...
3,1,total,daily,sales,sales,valid,28,1,1941,0.12,0.036405,0.000617,0.304209,/home/njimenez/Workspace/magister/forecast-tfm...
74,3,cat,daily,cum21,cumulative,valid,365,3,5760,0.38,0.037120,-0.023040,NaN,/home/njimenez/Workspace/magister/forecast-tfm...
4,1,total,daily,sales,sales,valid,35,1,1941,0.12,0.037460,0.000387,0.305394,/home/njimenez/Workspace/magister/forecast-tfm...
16,1,total,daily,cum14,cumulative,valid,365,1,1927,0.11,0.037576,0.023338,NaN,/home/njimenez/Workspace/magister/forecast-tfm...


# Modelo

In [17]:
df = pd.read_parquet('../artifacts/datasets/dataset_level_09_daily_store_dept.parquet')
df.head()

,agg_id,item_id,dept_id,cat_id,store_id,state_id,date,sales,avg_sell_price,price_lag_7,price_change,price_vs_mean,has_event,has_event_2,snap,event_name_1,event_type_1,event_name_2,event_type_2,year,month,day,dayofweek,weekofyear,is_weekend,cum7,cum14,cum21,cum28,cum35,cum42,cum365,price_vs_max,days_since_release,days_since_event,days_to_event,event_type_1_enc,event_type_2_enc,quarter,is_month_start,is_month_end,lag1,lag2,lag3,lag7,lag14,lag21,lag28,lag35,lag42,lag56,lag91,lag182,lag364,rolling_mean_lag7_window_size7,rolling_std_lag7_window_size7,rolling_min_lag7_window_size7,rolling_max_lag7_window_size7,rolling_mean_lag7_window_size14,rolling_std_lag7_window_size14,rolling_min_lag7_window_size14,rolling_max_lag7_window_size14,rolling_mean_lag28_window_size7,rolling_std_lag28_window_size7,rolling_min_lag28_window_size7,rolling_max_lag28_window_size7,rolling_mean_lag28_window_size28,rolling_std_lag28_window_size28,rolling_min_lag28_window_size28,rolling_max_lag28_window_size28,rolling_mean_lag28_window_size91,rolling_std_lag28_window_size91,rolling_min_lag28_window_size91,rolling_max_lag28_window_size91,rolling_mean_lag28_window_size7_truediv_rolling_mean_lag28_window_size28,rolling_mean_lag91_window_size28,rolling_std_lag91_window_size28,rolling_min_lag91_window_size28,rolling_max_lag91_window_size28,rolling_mean_lag91_window_size91,rolling_std_lag91_window_size91,rolling_min_lag91_window_size91,rolling_max_lag91_window_size91,expanding_mean_lag91,rolling_mean_lag364_window_size28,rolling_std_lag364_window_size28,rolling_min_lag364_window_size28,rolling_max_lag364_window_size28,rolling_mean_lag364_window_size91,rolling_std_lag364_window_size91,rolling_min_lag364_window_size91,rolling_max_lag364_window_size91,seasonal_rolling_mean_lag364_season_length7_window_size8,rolling_mean_lag365_window_size28,rolling_mean_lag365_window_size91
0,FOODS_1_FOODS_CA_1_CA,TOTAL,FOODS_1,FOODS,CA_1,CA,2011-01-29,297.0,2.293535,NaN,NaN,0.991603,0,0,0,NaN,NaN,NaN,NaN,2011,1,29,5,4,1,1533.0,3305.0,5047.0,6652.0,8373.0,10069.0,75935.0,1.000000,0,NaN,8.0,0,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FOODS_1_FOODS_CA_1_CA,TOTAL,FOODS_1,FOODS,CA_1,CA,2011-01-30,284.0,2.266514,NaN,NaN,0.979920,0,0,0,NaN,NaN,NaN,NaN,2011,1,30,6,4,1,1494.0,3467.0,5066.0,6632.0,8372.0,10089.0,75832.0,0.988218,1,NaN,7.0,0,0,1,0,0,297.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FOODS_1_FOODS_CA_1_CA,TOTAL,FOODS_1,FOODS,CA_1,CA,2011-01-31,214.0,2.108131,NaN,NaN,0.911444,0,0,0,NaN,NaN,NaN,NaN,2011,1,31,0,5,0,1456.0,3522.0,5080.0,6595.0,8342.0,10114.0,75824.0,0.919162,2,NaN,6.0,0,0,1,0,1,284.0,297.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FOODS_1_FOODS_CA_1_CA,TOTAL,FOODS_1,FOODS,CA_1,CA,2011-02-01,175.0,2.287943,NaN,NaN,0.989185,0,0,1,NaN,NaN,NaN,NaN,2011,2,1,1,5,0,1498.0,3516.0,5065.0,6696.0,8399.0,10125.0,75866.0,0.997562,3,NaN,5.0,0,0,1,1,0,214.0,284.0,297.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,FOODS_1_FOODS_CA_1_CA,TOTAL,FOODS_1,FOODS,CA_1,CA,2011-02-02,182.0,2.177802,NaN,NaN,0.941566,0,0,1,NaN,NaN,NaN,NaN,2011,2,2,2,5,0,1472.0,3498.0,5039.0,6717.0,8421.0,10161.0,75911.0,0.949539,4,NaN,4.0,0,0,1,0,0,175.0,214.0,284.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [24]:
TARGET = "sales"

ID_COLS = ["agg_id", "date"]

FEATURES = [c for c in df.columns if c not in ID_COLS + [TARGET]]

CATEGORICAL_FEATURES = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
]

NUMERICAL_FEATURES = [c for c in FEATURES if c not in CATEGORICAL_FEATURES]

FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

In [32]:
df = df.sort_values("date").copy()
df["date"] = pd.to_datetime(df["date"])

last_date = df["date"].max()

test_start = last_date - pd.Timedelta(days=364)
valid_start = test_start - pd.Timedelta(days=365 * 3)

train = df[df["date"] < valid_start]
valid = df[(df["date"] >= valid_start) & (df["date"] < test_start)]
test = df[df["date"] >= test_start]

X_train = train[FEATURES].copy()
y_train = train[TARGET]

X_valid = valid[FEATURES].copy()
y_valid = valid[TARGET]

X_test = test[FEATURES].copy()
y_test = test[TARGET]

for X in (X_train, X_valid, X_test):
    for col in CATEGORICAL_FEATURES:
        X[col] = X[col].astype("category")

In [35]:
from lightgbm import LGBMRegressor

model = LGBMRegressor(
    objective="regression",
    metric="rmse",
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(5),
    ],
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006758 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16958
[LightGBM] [Info] Number of data points in the train set: 33670, number of used features: 91
[LightGBM] [Info] Start training from score 395.830413
Training until validation scores don't improve for 50 rounds
[5]	valid_0's rmse: 382.961
[10]	valid_0's rmse: 238.364
[15]	valid_0's rmse: 160.034
[20]	valid_0's rmse: 125.887
[25]	valid_0's rmse: 112.773
[30]	valid_0's rmse: 109.141
[35]	valid_0's rmse: 108.638
[40]	valid_0's rmse: 108.94
[45]	valid_0's rmse: 108.913
[50]	valid_0's rmse: 108.382
[55]	valid_0's rmse: 108.152
[60]	valid_0's rmse: 108.353
[65]	valid_0's rmse: 108.616
[70]	valid_0's rmse: 108.993
[75]	valid_0's rmse: 108.868
[80]	valid_0's rmse: 109.185
[85]	valid_0's rmse: 109.491
[90]	valid_0's rmse: 109.473
[95]	valid_0's rmse: 109.54
[100]	valid_0's rmse: 109.688
Did not meet ear

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [36]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)

y_pred = model.predict(X_test)

print(f"MAE : {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.4f}")
print(f"R²  : {r2_score(y_test, y_pred):.4f}")

MAE : 65.5504
RMSE: 127.3044
R²  : 0.9573
